# Global Shipping Chokepoints — 01: Download and Load

Downloads two real, free, global datasets and loads them into Delta tables:

1. **World Bank Global Shipping Traffic Density** — a worldwide grid built from real
   AIS ship positions (Jan 2015 - Feb 2021), about 500m resolution at the equator, all vessel
   types combined (commercial, fishing, passenger, oil & gas, leisure). This is the real
   "did a ship actually pass through here" data.
   Catalog page: https://datacatalog.worldbank.org/search/dataset/0037580/global-shipping-traffic-density
2. **NGA World Port Index** — about 3,700 ports worldwide with name, country, and coordinates,
   used to label whatever chokepoints the density data turns up.
   Source used: https://hub.arcgis.com/datasets/EDT::world-port-index (CSV export)


## Step 0 — create the schema and volume

`/Volumes/workspace/global_shipping/raw_data` only becomes a valid path once the
`global_shipping` schema and its `raw_data` volume exist in Unity Catalog. Run this once.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.global_shipping;
CREATE VOLUME IF NOT EXISTS workspace.global_shipping.raw_data;

## Step 1 — download the shipping density raster

The World Bank catalog's download button for this dataset actually serves a ZIP
(`shipdensity_global.zip`, ~511 MB), containing the **all-vessel-types** combined
density layer — not the commercial-only layer the catalog page also lists separately.
We're using the combined layer: the major commercial chokepoints (Suez, Malacca,
Panama, Hormuz) dominate it regardless, and it saves a second multi-hundred-MB download.
Anything we publish from this will say "all vessel types," not "commercial only."

In [0]:
%sh
cd /Volumes/workspace/global_shipping/raw_data
wget -O shipdensity_global.zip "https://datacatalogfiles.worldbank.org/ddh-published/0037580/5/DR0045406/shipdensity_global.zip"
unzip -o shipdensity_global.zip -d shipdensity_extracted
ls -lh shipdensity_extracted

--2026-09-12 23:42:37--  https://datacatalogfiles.worldbank.org/ddh-published/0037580/5/DR0045406/shipdensity_global.zip
Resolving datacatalogfiles.worldbank.org (datacatalogfiles.worldbank.org)... 150.171.110.193, 2620:1ec:48:1::40, 2620:1ec:29:1::40
Connecting to datacatalogfiles.worldbank.org (datacatalogfiles.worldbank.org)|150.171.110.193|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 534907276 (510M) [application/octet-stream]
Saving to: ‘shipdensity_global.zip’

     0K .......... .......... .......... .......... ..........  0% 2.65M 3m13s
    50K .......... .......... .......... .......... ..........  0% 4.03M 2m40s
   100K .......... .......... .......... .......... ..........  0% 4.06M 2m28s
   150K .......... .......... .......... .......... ..........  0% 3.77M 2m25s
   200K .......... .......... .......... .......... ..........  0% 4.98M 2m16s
   250K .......... .......... .......... .......... ..........  0% 13.1M 2m0s
   300K .......... .......

In [0]:
%pip install rasterio
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import rasterio

RASTER_PATH = "/Volumes/workspace/global_shipping/raw_data/shipdensity_extracted/shipdensity_global.tif"

with rasterio.open(RASTER_PATH) as src:
    print("CRS:", src.crs)
    print("Size (width x height):", src.width, "x", src.height)
    print("Bounds:", src.bounds)
    print("Native pixel size (degrees):", src.res)
    print("Overview levels already built into the file:", src.overviews(1))

CRS: EPSG:4326
Size (width x height): 72006 x 33998
Bounds: BoundingBox(left=-180.015311275, bottom=-84.98735206299992, right=180.01468872500004, top=85.00264793700009)
Native pixel size (degrees): (0.005, 0.005)
Overview levels already built into the file: [2, 4, 8, 16, 32, 64, 128, 255]


## Downsample to a Spark-friendly grid

The native grid is far too fine (billions of cells at about 500m resolution, mostly zero —
land, or open ocean nobody crosses) to reason about globally, and the uncompressed file
is 9.2 GB. Instead of reading every native pixel in Python, we ask GDAL (via rasterio)
to decimate directly to a 0.05-degree grid (about 5.5 km at the equator) using `Resampling.sum`
— it adds up the native cells inside each new, coarser cell, which is the correct way to
combine a *count/density* layer (unlike `average`, which would understate how much
traffic passes through a busy coarse cell). GDAL does this fast using the overview
pyramid already built into the file. Only non-zero cells are kept afterward.


In [0]:
import numpy as np
import pandas as pd
from rasterio.enums import Resampling
from rasterio.transform import Affine

BLOCK = 10  # 10 native pixels/side -> ~0.05-degree cells if native res is ~0.005 degree (confirmed above)

with rasterio.open(RASTER_PATH) as src:
    out_height = src.height // BLOCK
    out_width = src.width // BLOCK
    # Resampling.sum only works for warp operations, not plain reads. Every block here is
    # exactly BLOCK x BLOCK, so average * (BLOCK*BLOCK) gives the same result as a true sum,
    # without needing the warp API. Reading as float64 avoids premature rounding.
    data = src.read(1, out_shape=(out_height, out_width), resampling=Resampling.average, out_dtype="float64")
    data = data * (BLOCK * BLOCK)
    out_transform = src.transform * Affine.scale(src.width / out_width, src.height / out_height)

nz_rows, nz_cols = np.nonzero(data)
lons, lats = rasterio.transform.xy(out_transform, nz_rows, nz_cols)

pdf = pd.DataFrame({
    "lat": lats,
    "lon": lons,
    "traffic_density": np.round(data[nz_rows, nz_cols]).astype("int64"),
})
print(f"{len(pdf):,} non-zero 0.05-degree cells kept (out of ~{out_height * out_width:,} possible globally)")

sdf = spark.createDataFrame(pdf)
sdf.write.format("delta").mode("overwrite").saveAsTable("workspace.global_shipping.traffic_density_grid")
display(sdf.orderBy(sdf.traffic_density.desc()).limit(20))

3,040,763 non-zero 0.05-degree cells kept (out of ~24,472,800 possible globally)


lat,lon,traffic_density
52.46999274429635,4.475061641666656,4976479100
55.52071060248994,10.875594974999984,4956071800
57.7712401700098,10.77558664166665,4937670000
55.77076944332548,12.725749141666654,4896435000
57.571193097341364,11.675661641666665,4869315700
62.47234637771795,6.275211641666658,4854910300
69.37397038477884,18.12619914166666,4849296700
63.17251113205746,7.87534497499999,4839590500
57.721228401842694,10.77558664166665,4805092800
55.8707929796597,12.775753308333321,4798376100


## Step 2 — load the World Port Index

The ArcGIS Hub CSV download button doesn't give a stable, copyable URL — it just
downloads straight to your computer. So upload the file by hand instead of `wget`:

1. In the left sidebar, go to **Catalog**.
2. Navigate to **workspace → global_shipping → raw_data** (the volume created in Step 0).
3. Click **Upload to this volume**, and select the CSV file you already downloaded
   from ArcGIS Hub (check your Downloads folder — something like `World_Port_Index.csv`).
4. Once it finishes uploading, run the cell below. If the uploaded filename is
   different from `world_port_index.csv`, update `PORT_CSV_PATH` to match.

In [0]:
%sh
ls -la /Volumes/workspace/global_shipping/raw_data/


total 523111
drwxrwxrwx 2 nobody                           nogroup      4096 Sep 12 23:59 .
drwxrwxrwx 2 nobody                           nogroup      4096 Sep 12 23:59 ..
drwxrwxrwx 2 spark-ad52eb11-13fb-4fbf-8e34-5d nogroup      4096 Sep 12 23:44 shipdensity_extracted
-rwxrwxrwx 1 spark-ad52eb11-13fb-4fbf-8e34-5d nogroup 534907276 Sep 12 23:42 shipdensity_global.zip
-rwxrwxrwx 1 nobody                           nogroup    745694 Sep 12 23:59 World_Port_Index.csv


In [0]:
PORT_CSV_PATH = "/Volumes/workspace/global_shipping/raw_data/World_Port_Index.csv"

ports = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(PORT_CSV_PATH)
)
print(f"{ports.count():,} ports loaded")
ports.write.format("delta").mode("overwrite").saveAsTable("workspace.global_shipping.world_port_index")
display(ports.limit(10))


3,669 ports loaded


FID,INDEX_NO,REGION_NO,PORT_NAME,COUNTRY,LATITUDE,LONGITUDE,LAT_DEG,LAT_MIN,LAT_HEMI,LONG_DEG,LONG_MIN,LONG_HEMI,PUB,CHART,HARBORSIZE,HARBORTYPE,SHELTER,ENTRY_TIDE,ENTRYSWELL,ENTRY_ICE,ENTRYOTHER,OVERHD_LIM,CHAN_DEPTH,ANCH_DEPTH,CARGODEPTH,OIL_DEPTH,TIDE_RANGE,MAX_VESSEL,HOLDGROUND,TURN_BASIN,PORTOFENTR,US_REP,ETAMESSAGE,PILOT_REQD,PILOTAVAIL,LOC_ASSIST,PILOTADVSD,TUGSALVAGE,TUG_ASSIST,PRATIQUE,SSCC_CERT,QUAR_OTHER,COMM_PHONE,COMM_FAX,COMM_RADIO,COMM_VHF,COMM_AIR,COMM_RAIL,CARGOWHARF,CARGO_ANCH,CARGMDMOOR,CARBCHMOOR,CARICEMOOR,MED_FACIL,GARBAGE,DEGAUSS,DRTYBALLST,CRANEFIXED,CRANEMOBIL,CRANEFLOAT,LIFT_100_,LIFT50_100,LIFT_25_49,LIFT_0_24,LONGSHORE,ELECTRICAL,SERV_STEAM,NAV_EQUIP,ELECREPAIR,PROVISIONS,WATER,FUEL_OIL,DIESEL,DECKSUPPLY,ENG_SUPPLY,REPAIRCODE,DRYDOCK,RAILWAY
1,44730,44360,AYVALIK,TR,39.316667,26.7,39,19,N,26,42,E,132,54382,V,CN,G,N,N,N,Y,,N,K,O,,2,M,Y,,N,N,N,,,,,,,,,,,Y,,,,,,,,,,,,,N,,,,,,,,,,,,,Y,Y,,,,,,,
2,44750,44360,IZMIR,TR,38.433333,27.133333,38,26,N,27,8,E,132,54387,L,CN,F,N,Y,N,Y,,K,G,L,N,2,M,Y,Y,Y,Y,Y,Y,Y,,Y,,Y,Y,Y,Y,Y,,Y,Y,Y,Y,Y,Y,,,,Y,Y,,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,N,N,,Y,Y,Y,Y,Y,Y,B,M,S
3,57610,57510,VINH CAM RANH,VN,11.883333,109.166667,11,53,N,109,10,E,161,93446,S,CN,G,N,N,N,Y,,L,E,N,H,2,L,,,Y,N,Y,Y,Y,,Y,,Y,,,,Y,Y,,,Y,,Y,Y,,,,Y,,,N,,,Y,,Y,,,,,,,,,Y,,,,,,,
4,57620,57510,NHA TRANG,VN,12.25,109.233333,12,15,N,109,14,E,161,93442,S,OR,F,Y,Y,N,Y,N,J,G,J,K,2,M,Y,Y,Y,N,Y,Y,Y,,Y,,Y,Y,Y,Y,Y,Y,Y,,Y,,Y,Y,,,,Y,Y,,N,Y,Y,N,Y,Y,Y,Y,Y,N,N,N,Y,Y,Y,Y,,N,N,C,L,
5,57640,57510,QUI NHON,VN,13.766667,109.233333,13,46,N,109,14,E,161,93491,V,CN,F,Y,Y,N,Y,N,J,J,M,L,2,M,Y,Y,Y,N,Y,Y,Y,,Y,,Y,Y,Y,Y,,,Y,Y,Y,Y,Y,,,,,Y,,,N,Y,Y,,,Y,Y,Y,Y,Y,N,N,N,Y,Y,Y,Y,N,N,C,M,S
6,57650,57510,DA NANG,VN,16.1,108.216667,16,6,N,108,13,E,161,93512,S,RN,F,Y,Y,N,N,Y,H,F,J,H,1,M,Y,Y,Y,N,Y,Y,,,,N,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,,,,,Y,Y,,Y,,Y,,,Y,Y,Y,Y,N,N,Y,Y,,Y,Y,Y,,,C,,S
7,57680,57510,HAI PHONG,VN,20.916667,106.683333,20,55,N,106,41,E,161,93032,S,RN,G,Y,Y,N,Y,Y,M,J,K,O,4,M,Y,Y,Y,N,Y,Y,Y,,Y,,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,,,Y,Y,,Y,Y,Y,Y,,Y,Y,Y,Y,,,,,,Y,Y,Y,Y,Y,B,M,S
8,57700,57510,HON GAI,VN,20.95,107.066667,20,57,N,107,4,E,161,93626,V,OR,F,Y,N,N,Y,,L,F,J,K,2,M,,Y,Y,N,Y,Y,Y,,Y,,Y,Y,Y,Y,Y,Y,Y,,,Y,Y,,,,,Y,Y,,N,,Y,Y,,Y,Y,Y,Y,,,,,Y,Y,Y,Y,,,C,,S
9,57070,56940,YANDINA,SB,-9.083333,159.216667,9,5,S,159,13,E,126,82384,V,CN,G,N,N,N,Y,,A,A,M,,1,M,Y,,Y,,Y,N,Y,,Y,N,N,,,Y,,,Y,Y,Y,,Y,,,,,,N,,N,,,,,,,,Y,Y,N,N,Y,Y,Y,N,Y,N,N,C,,
10,56323,56050,TERN ISLAND,US,23.866667,-166.283333,23,52,N,166,17,W,CP07,19402,V,CN,P,N,N,N,Y,,N,A,N,,1,,Y,,,,,,,,,,,,,,,Y,,,,,,,,,,Y,,,,,,,,,,,,,,,,,Y,,,,,C,,
